In [113]:
from langchain_cohere import ChatCohere
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv
import requests

In [114]:
#tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    url = f"https://v6.exchangerate-api.com/v6/ea2d0d542d7ed3cef8a8f8e7/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()


@tool
def convert(base_currency_value: int, conversion_rate:Annotated[float,InjectedToolArg]) -> float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    return base_currency_value * conversion_rate

In [115]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'NPR'})['conversion_rate']

143.8529

In [116]:
convert.invoke({'base_currency_value':10000,'conversion_rate':143.8529})

1438529.0

In [117]:
#tool binding
load_dotenv()
llm = ChatCohere(model="command-a-03-2025")

In [118]:
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])

In [119]:

system_msg = SystemMessage(
    content=(
        "You are allowed to output multiple tool calls in a single response when appropriate. "
        "If you need to compute something that requires two steps (for example: fetch a conversion rate, "
        "then convert an amount), you may output BOTH tool calls in the same response. "
        "If you do not know an exact numeric value (like the conversion rate), you may include a reasonable "
        "estimate in the second tool call argument named 'conversion_rate'. Mark estimates clearly in text if possible. "
        "Always provide the plan and the sequence of tool calls before the tool calls."
    )
)
human_message=HumanMessage("What is the conversion factor between USD and NPR, and based on that you can convert 10 usd to nrp")
messages=[system_msg,human_message]

In [120]:
messages

[SystemMessage(content="You are allowed to output multiple tool calls in a single response when appropriate. If you need to compute something that requires two steps (for example: fetch a conversion rate, then convert an amount), you may output BOTH tool calls in the same response. If you do not know an exact numeric value (like the conversion rate), you may include a reasonable estimate in the second tool call argument named 'conversion_rate'. Mark estimates clearly in text if possible. Always provide the plan and the sequence of tool calls before the tool calls.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the conversion factor between USD and NPR, and based on that you can convert 10 usd to nrp', additional_kwargs={}, response_metadata={})]

In [121]:
ai_messages = llm_with_tools.invoke(messages)

In [122]:
messages.append(ai_messages)

In [123]:
ai_messages.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'NPR'},
  'id': 'get_conversion_factor_yn8bzjct3ren',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'convert_400r7f47f90e',
  'type': 'tool_call'}]

In [124]:
import json

for tool_call in ai_messages.tool_calls:
    # execute the 1st tool and get the conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        #fetch this conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        # append this tool message to message list
        messages.append(tool_message1)
    # execute the 2nd tool using the conversion rate from the tool 1
    if tool_call['name'] == 'convert':
        # fetch the current arg
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [125]:
messages

[SystemMessage(content="You are allowed to output multiple tool calls in a single response when appropriate. If you need to compute something that requires two steps (for example: fetch a conversion rate, then convert an amount), you may output BOTH tool calls in the same response. If you do not know an exact numeric value (like the conversion rate), you may include a reasonable estimate in the second tool call argument named 'conversion_rate'. Mark estimates clearly in text if possible. Always provide the plan and the sequence of tool calls before the tool calls.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the conversion factor between USD and NPR, and based on that you can convert 10 usd to nrp', additional_kwargs={}, response_metadata={}),
 AIMessage(content='I will first find the conversion factor between USD and NPR. Then, I will use this factor to convert 10 USD to NPR.', additional_kwargs={'id': '4813498f-ca55-4dc1-8390-33cbf06aa2e6', 'finish_re

In [126]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and NPR is 143.8529.\n\nBased on this, 10 USD is equal to 1438.529 NPR.'